# SmartVision AI — Phase 3: YOLOv8 object detection (25 classes)

Fine-tune **YOLOv8s** (COCO-pretrained) on the 25-class subset.

**Runtime:** Colab **T4 GPU**.

**Input:** `smartvision_dataset/detection/` with separate `images/{train,val,test}` and `labels/{train,val,test}` plus `data.yaml` (`nc: 25`).

**Output:** `models/yolov8_best.pt`, `reports/yolo_metrics.json`, prediction mosaics, failure gallery.

Rubric floor: **mAP@0.5 > 75%**.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install ultralytics pyyaml pandas matplotlib seaborn pillow opencv-python-headless

In [ ]:
import os, sys, json, time, random, shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/Smart_Vision_AI")
    PROJECT_ROOT.mkdir(exist_ok=True)
    zip_path = Path("/content/drive/MyDrive/smartvision_dataset.zip")
    data_dir = PROJECT_ROOT / "smartvision_dataset"
    if zip_path.exists() and not (data_dir / "detection" / "data.yaml").exists():
        import zipfile
        print("Unzipping...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(PROJECT_ROOT)
    OUT_ROOT = Path("/content/drive/MyDrive/SmartVision_artifacts")
else:
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    OUT_ROOT = PROJECT_ROOT

DET = PROJECT_ROOT / "smartvision_dataset" / "detection"
if not (DET / "data.yaml").exists():
    alt = Path("/content/smartvision_dataset/detection")
    if (alt / "data.yaml").exists():
        DET = alt
print("DET =", DET)

MODELS_DIR = OUT_ROOT / "models"
FIGURES = OUT_ROOT / "reports" / "figures"
REPORTS = OUT_ROOT / "reports"
for p in (MODELS_DIR, FIGURES, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

yaml_text = (DET / "data.yaml").read_text(encoding="utf-8")
print(yaml_text)

# Rewrite path: to the actual DET location (Colab unzip may differ from the machine that wrote yaml)
import yaml
cfg = yaml.safe_load(yaml_text)
cfg["path"] = str(DET.resolve())
# force 25 classes
assert int(cfg.get("nc", 0)) == 25, cfg
names = cfg["names"]
if isinstance(names, dict):
    class_names = [names[i] for i in range(len(names))]
else:
    class_names = list(names)
assert "train" not in class_names, class_names
assert len(class_names) == 25
(DET / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Updated data.yaml path ->", cfg["path"])
print("names:", class_names)

In [ ]:
## Verify YOLO labels against images (data-driven sanity, not assumed)

def verify_split(split):
    img_dir = DET / "images" / split
    lab_dir = DET / "labels" / split
    images = sorted(img_dir.glob("*.jpg"))
    missing, bad, objs = [], 0, 0
    per_class = Counter()
    for imgp in images:
        lab = lab_dir / (imgp.stem + ".txt")
        if not lab.exists():
            missing.append(imgp.name)
            continue
        for line in lab.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                bad += 1
                continue
            cls, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
            if not (0 <= cls < 25):
                bad += 1
                continue
            if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
                bad += 1
                continue
            objs += 1
            per_class[class_names[cls]] += 1
    return {"n_images": len(images), "n_labels": len(list(lab_dir.glob("*.txt"))),
            "missing_labels": len(missing), "bad_rows": bad, "objects": objs, "per_class": dict(per_class)}

for split in ("train", "val", "test"):
    stats = verify_split(split)
    print(split, {k: stats[k] for k in stats if k != "per_class"})
print("\\nTrain objects per class:")
print(pd.Series(verify_split("train")["per_class"]).reindex(class_names))

In [ ]:
## Train YOLOv8s (COCO pretrained, 25-class head re-init)

from ultralytics import YOLO

model = YOLO("yolov8s.pt")
results = model.train(
    data=str(DET / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=12,
    seed=42,
    workers=2,
    project=str(OUT_ROOT / "yolo_runs"),
    name="smartvision_yolov8s",
    exist_ok=True,
    pretrained=True,
    optimizer="AdamW",
    lr0=0.001,
    plots=True,
    verbose=True,
)
print(results)

In [ ]:
## Evaluate on val (primary) and test (held-out report)

best = Path(OUT_ROOT / "yolo_runs" / "smartvision_yolov8s" / "weights" / "best.pt")
print("best exists", best.exists(), best)
trained = YOLO(str(best))

val_metrics = trained.val(data=str(DET / "data.yaml"), split="val", plots=True)
print("VAL", val_metrics.results_dict if hasattr(val_metrics, "results_dict") else val_metrics)

test_metrics = trained.val(data=str(DET / "data.yaml"), split="test", plots=True)
print("TEST", test_metrics.results_dict if hasattr(test_metrics, "results_dict") else test_metrics)

def extract(m):
    d = m.results_dict if hasattr(m, "results_dict") else {}
    # ultralytics keys: metrics/mAP50(B), metrics/mAP50-95(B), metrics/precision(B), metrics/recall(B)
    def g(*keys):
        for k in keys:
            if k in d:
                return float(d[k])
        box = getattr(m, "box", None)
        if box is not None:
            for attr in keys:
                if hasattr(box, attr.split("/")[-1].split("(")[0]):
                    pass
        return float(getattr(getattr(m, "box", m), "map50", d.get("metrics/mAP50(B)", 0.0)) or 0.0)
    return {
        "map50": float(d.get("metrics/mAP50(B)", getattr(getattr(m, "box", None), "map50", 0.0) or 0.0)),
        "map50_95": float(d.get("metrics/mAP50-95(B)", getattr(getattr(m, "box", None), "map", 0.0) or 0.0)),
        "precision": float(d.get("metrics/precision(B)", getattr(getattr(m, "box", None), "mp", 0.0) or 0.0)),
        "recall": float(d.get("metrics/recall(B)", getattr(getattr(m, "box", None), "mr", 0.0) or 0.0)),
        "raw": {k: float(v) if isinstance(v, (int, float, np.floating)) else str(v) for k, v in d.items()},
    }

val_ex = extract(val_metrics)
test_ex = extract(test_metrics)
print("VAL extracted", {k: val_ex[k] for k in ("map50", "map50_95", "precision", "recall")})
print("TEST extracted", {k: test_ex[k] for k in ("map50", "map50_95", "precision", "recall")})

# Per-class AP if available
ap_per_class = {}
box = getattr(val_metrics, "box", None)
if box is not None and hasattr(box, "ap_class_index") and hasattr(box, "ap50"):
    for idx, ap in zip(box.ap_class_index, box.ap50):
        ap_per_class[class_names[int(idx)]] = float(ap)
print("per-class AP50", ap_per_class)

In [ ]:
## Speed (FPS) on a handful of val images

import time
val_imgs = sorted((DET / "images" / "val").glob("*.jpg"))[:50]
# warmup
_ = trained.predict(source=str(val_imgs[0]), verbose=False)
t0 = time.perf_counter()
_ = trained.predict(source=[str(p) for p in val_imgs], verbose=False)
dt = time.perf_counter() - t0
fps = len(val_imgs) / max(dt, 1e-6)
print(f"{len(val_imgs)} images in {dt:.2f}s -> {fps:.1f} FPS")

In [ ]:
## Visualize predictions on sample val images

samples = val_imgs[:8]
pred = trained.predict(source=[str(p) for p in samples], conf=0.5, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, r, p in zip(axes.ravel(), pred, samples):
    plotted = r.plot()  # BGR ndarray
    ax.imshow(plotted[:, :, ::-1])
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")
fig.suptitle("YOLOv8 val predictions (conf>=0.5)")
fig.tight_layout()
fig.savefig(FIGURES / "yolo_val_samples.png", dpi=140)
plt.show()

In [ ]:
## Failure gallery: images with 0 detections or low mean confidence at conf=0.5

fail = []
ok = []
all_val = sorted((DET / "images" / "val").glob("*.jpg"))
chunk = all_val[:120]
res_all = trained.predict(source=[str(p) for p in chunk], conf=0.5, verbose=False)
for r, p in zip(res_all, chunk):
    n = 0 if r.boxes is None else len(r.boxes)
    confs = [] if n == 0 else [float(c) for c in r.boxes.conf]
    mean_c = float(np.mean(confs)) if confs else 0.0
    rec = {"path": p, "n": n, "mean_conf": mean_c}
    if n == 0:
        fail.append(rec)
    else:
        ok.append(rec)

print(f"scanned {len(chunk)} val images; zero-detection={len(fail)}")
# also take lowest-confidence successes as near-failures
ok.sort(key=lambda d: d["mean_conf"])
gallery = fail[:8] + ok[: max(0, 8 - len(fail[:8]))]
if gallery:
    shown = trained.predict(source=[str(d["path"]) for d in gallery], conf=0.25, verbose=False)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, r, rec in zip(axes.ravel(), shown, gallery):
        ax.imshow(r.plot()[:, :, ::-1])
        ax.set_title(f"n={rec['n']} conf={rec['mean_conf']:.2f}", fontsize=8)
        ax.axis("off")
    fig.suptitle("Failure / low-confidence cases (drawn at conf=0.25 so missed boxes can appear)")
    fig.tight_layout()
    fig.savefig(FIGURES / "yolo_failures.png", dpi=140)
    plt.show()
else:
    print("No failures in the scanned slice.")

In [ ]:
## Copy best weights + write yolo_metrics.json
# If mAP50 on VAL is still <= 0.75, train longer (based on this run, not a guess).

map50 = val_ex["map50"]
print("val mAP50 =", map50)
if map50 <= 0.75:
    print("Below 75% floor — continuing training 30 more epochs from best.pt")
    model2 = YOLO(str(best))
    model2.train(
        data=str(DET / "data.yaml"),
        epochs=30,
        imgsz=640,
        batch=16,
        patience=10,
        seed=42,
        workers=2,
        project=str(OUT_ROOT / "yolo_runs"),
        name="smartvision_yolov8s_more",
        exist_ok=True,
        optimizer="AdamW",
        lr0=0.0005,
        plots=True,
    )
    best = Path(OUT_ROOT / "yolo_runs" / "smartvision_yolov8s_more" / "weights" / "best.pt")
    trained = YOLO(str(best))
    val_metrics = trained.val(data=str(DET / "data.yaml"), split="val", plots=True)
    val_ex = extract(val_metrics)
    print("VAL after extra train", {k: val_ex[k] for k in ("map50", "map50_95", "precision", "recall")})

dest = MODELS_DIR / "yolov8_best.pt"
shutil.copy2(best, dest)
print("Copied", dest)

yolo_payload = {
    "model": "YOLOv8s",
    "weights": str(dest),
    "val": {k: v for k, v in val_ex.items() if k != "raw"},
    "test": {k: v for k, v in test_ex.items() if k != "raw"},
    "fps": float(fps),
    "per_class_ap50": ap_per_class,
    "n_classes": 25,
    "class_names": class_names,
    "meets_map50_floor": bool(val_ex["map50"] > 0.75),
}
(REPORTS / "yolo_metrics.json").write_text(json.dumps(yolo_payload, indent=2), encoding="utf-8")
print(json.dumps(yolo_payload, indent=2)[:1500])